# 模拟题 B — Feature Selection: E-commerce Product Reviews
## 进阶版（更接近期末考试风格）

**关于数据集：** `ecommerce_reviews.csv`

该数据集包含 8000 条电商平台商品评价，目标是预测用户对商品的整体评分。

**变量说明：**
- `user_id`: 用户 ID（删除）
- `category`: 商品类别（删除）
- `verified_purchase`: 是否验证购买（删除）
- `price`: 商品价格（美元）
- `delivery_days`: 配送天数
- `return_rate`: 该商品历史退货率（%）
- `product_quality`: 商品质量评分 (1–5)
- `packaging`: 包装评分 (1–5)
- `description_accuracy`: 商品描述准确性评分 (1–5)
- `seller_response`: 卖家响应速度评分 (1–5)
- `value_score`: 性价比评分 (1–5)
- `ease_of_use`: 易用性评分 (1–5)
- `repeat_purchase_intent`: 复购意愿 (1–5)
- `overall_score`: 整体评分 (1–5)【目标变量】

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# B1 ──────────────────────────────────────────────────────────────────
# 读入 ecommerce_reviews.csv，删除 user_id, category, verified_purchase。
# 以 overall_score 为分层变量，80/20 分层分割，random_state=777。
#
# (a) train 样本中 price 的平均值是多少？
# (b) train 和 test 中 overall_score 的均值是否相近？（验证分层是否有效）

# data = pd.read_csv('ecommerce_reviews.csv')
# data2 = data.drop(columns=['user_id', 'category', 'verified_purchase'])
# y = data2['overall_score']
# X = data2.drop('overall_score', axis=1)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, train_size=0.8, random_state=777, stratify=y)
# print(f'Train price mean: {round(X_train["price"].mean(), 2)}')
# print(f'Train y mean: {round(y_train.mean(), 4)}')
# print(f'Test  y mean: {round(y_test.mean(), 4)}')

In [ ]:
# B2 ──────────────────────────────────────────────────────────────────
# 计算所有预测变量与 overall_score 的绝对相关系数，升序排列。
# (a) 相关性最弱的变量是哪个？
# (b) 相关性最强的变量是哪个？
# (c) 仅凭相关系数就可以决定是否保留变量吗？为什么？

# train_full = X_train.copy()
# train_full['overall_score'] = y_train.values
# corr = train_full.corr()['overall_score'].drop('overall_score').abs().sort_values()
# print(corr)

# 📌 概念题答案提示：
# 相关系数只衡量双变量关系，忽略了多变量联合效应。
# 低相关的变量在控制其他变量后可能仍有增量解释力；
# 高相关的两个变量可能存在多重共线性 → 需要 VIF 进一步诊断。

In [ ]:
# B3 ──────────────────────────────────────────────────────────────────
# 用 statsmodels OLS 对所有变量同时建模。
# (a) 哪些变量显著？（p < 0.05）
# (b) 模型的 R² 是多少？
# (c) 与双变量相关系数的结论相比，有何不同？

# X_train_sm = sm.add_constant(X_train)
# ols = sm.OLS(y_train, X_train_sm).fit()
# print(ols.summary())
# sig = ols.pvalues[ols.pvalues < 0.05].drop('const', errors='ignore')
# print('显著变量:', sig.index.tolist())

In [ ]:
# B4 ──────────────────────────────────────────────────────────────────
# 计算 VIF，判断是否存在多重共线性。
# 如果存在 VIF > 5 的变量，应该怎么处理？

# vif_data = pd.DataFrame()
# vif_data['feature'] = X_train.columns
# vif_data['VIF'] = [variance_inflation_factor(X_train.values.astype(float), i)
#                    for i in range(X_train.shape[1])]
# print(vif_data.sort_values('VIF', ascending=False).round(3))

# 📌 如果 VIF > 5：
# 选项1：删除相关性最高的冗余变量
# 选项2：使用 PCA 将相关变量压缩为主成分
# 选项3：使用 Ridge/Lasso 正则化（对多重共线性更稳健）

In [ ]:
# B5 ──────────────────────────────────────────────────────────────────
# 用 Forward 和 Backward SequentialFeatureSelector（scoring='r2', cv=5）
# 分别选择变量。
# (a) Forward 选中了哪些变量？
# (b) Backward 选中了哪些变量？
# (c) 两种方法是否选出了相同的变量子集？如果不同，如何决定使用哪个？

# lr = LinearRegression()
# sfs_fwd = SequentialFeatureSelector(lr, n_features_to_select='auto',
#                                      direction='forward', scoring='r2', cv=5)
# sfs_fwd.fit(X_train, y_train)
# fwd = X_train.columns[sfs_fwd.get_support()].tolist()

# sfs_bwd = SequentialFeatureSelector(lr, n_features_to_select='auto',
#                                      direction='backward', scoring='r2', cv=5)
# sfs_bwd.fit(X_train, y_train)
# bwd = X_train.columns[sfs_bwd.get_support()].tolist()

# print('Forward:', fwd)
# print('Backward:', bwd)

In [ ]:
# B6 ──────────────────────────────────────────────────────────────────
# 用 StandardScaler 标准化，再用 LassoCV（cv=5）选择变量。
# (a) 最优 alpha 是多少？
# (b) 哪些变量系数被缩减为 0？
# (c) Lasso 与 Forward Selection 的结果是否一致？

# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled  = scaler.transform(X_test)
# lasso_cv = LassoCV(cv=5, random_state=777)
# lasso_cv.fit(X_train_scaled, y_train)
# coef = pd.Series(lasso_cv.coef_, index=X_train.columns)
# print(f'Best alpha: {lasso_cv.alpha_:.6f}')
# print(coef.sort_values())
# print('缩减为 0:', coef[coef == 0].index.tolist())

In [ ]:
# B7 ──────────────────────────────────────────────────────────────────
# 分别用以下三个模型预测 test 样本的 overall_score，比较 R²：
# (a) 全变量线性回归（sklearn LinearRegression）
# (b) Lasso 选中变量的线性回归
# (c) Forward Selection 选中变量的线性回归
# 哪个模型 test R² 最高？这说明了什么？

# lasso_selected = coef[coef != 0].index.tolist()

# # 全变量
# lr_all = LinearRegression().fit(X_train, y_train)
# r2_all = r2_score(y_test, lr_all.predict(X_test))

# # Lasso 选中变量
# lr_lasso = LinearRegression().fit(X_train[lasso_selected], y_train)
# r2_lasso = r2_score(y_test, lr_lasso.predict(X_test[lasso_selected]))

# # Forward 选中变量
# lr_fwd = LinearRegression().fit(X_train[fwd], y_train)
# r2_fwd = r2_score(y_test, lr_fwd.predict(X_test[fwd]))

# print(f'全变量 R²:  {round(r2_all, 4)}')
# print(f'Lasso R²:   {round(r2_lasso, 4)}')
# print(f'Forward R²: {round(r2_fwd, 4)}')

In [ ]:
# B8 ──────────────────────────────────────────────────────────────────
# 使用 PCA 对标准化后的训练数据降维。
# (a) 打印各主成分数量对应的累积解释方差百分比。
# (b) 若需保留约 70% (±2%) 的方差，需要几个主成分？
# (c) 用该数量的主成分拟合线性回归，报告 train R² 和 test R²。
# (d) 与 Lasso LR 相比，PCA LR 的 R² 更高还是更低？尝试解释原因。

# pca_full = PCA()
# pca_full.fit(X_train_scaled)
# cumvar = np.cumsum(pca_full.explained_variance_ratio_)
# for i, v in enumerate(cumvar):
#     print(f'{i+1} components: {round(v*100,2)}%')

# n_comp = ???  # 根据上面结果填入
# pca = PCA(n_components=n_comp)
# X_tr_pca = pca.fit_transform(X_train_scaled)
# X_te_pca = pca.transform(X_test_scaled)
# lr_pca = LinearRegression().fit(X_tr_pca, y_train)
# print('PCA Train R²:', round(r2_score(y_train, lr_pca.predict(X_tr_pca)), 4))
# print('PCA Test  R²:', round(r2_score(y_test,  lr_pca.predict(X_te_pca)), 4))

# 📌 原因提示：PCA 的主成分是方差最大的方向，不一定是与目标变量最相关的方向。
# 而 Lasso 直接优化预测精度，因此通常 R² 更高。

In [ ]:
# B9 ──────────────────────────────────────────────────────────────────
# 【综合概念题】
# 某同学使用相关系数筛选掉了与 overall_score 相关性最弱的两个变量，
# 然后直接用剩余变量拟合线性回归。
# 请评价这个做法，并说明更好的替代方案。

# 答案框架：
# 问题1：相关系数只看双变量关系，控制其他变量后结论可能不同
#         → 应该用 OLS 的 p-value 或 VIF 来判断多变量下的重要性
# 问题2：手动删除变量没有利用到数据驱动的最优子集搜索
#         → 应该用 Forward/Backward Selection 或 Lasso
# 问题3：没有考虑变量间的多重共线性
#         → 应该先计算 VIF
# 更好方案：
#   Step 1: 全变量 OLS → 看 p-value 和 R²
#   Step 2: VIF → 检查多重共线性
#   Step 3: Lasso / Forward Selection → 自动选择最优子集
#   Step 4: 在 test 集上验证 R²

In [ ]:
# B10 ─────────────────────────────────────────────────────────────────
# 【代码填空题】以下代码有 3 处错误，请找出并改正。

# 错误代码：
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled  = scaler.fit_transform(X_test)        # 错误1

# pca = PCA(n_components=5)
# X_train_pca = pca.fit_transform(X_train_scaled)
# X_test_pca  = pca.fit_transform(X_test_scaled)        # 错误2

# lr = LinearRegression()
# lr.fit(X_train_pca, y_train)
# r2 = r2_score(lr.predict(X_test_pca), y_test)         # 错误3

# 正确代码：
# 错误1修正：X_test_scaled = scaler.transform(X_test)
#   → test 只能用 train 拟合的 scaler 来转换，不能重新 fit
# 错误2修正：X_test_pca = pca.transform(X_test_scaled)
#   → 同理，PCA 也只 fit train，transform test
# 错误3修正：r2 = r2_score(y_test, lr.predict(X_test_pca))
#   → r2_score(y_true, y_pred)，参数顺序不能颠倒